## 生成式对话机器人

In [2]:
from datasets import Dataset
from transformers import AutoTokenizer,AutoModelForCausalLM,DataCollatorForSeq2Seq,TrainingArguments,Trainer

In [3]:
ds = Dataset.load_from_disk("./alpaca_data_zh")

In [4]:
ds

Dataset({
    features: ['output', 'input', 'instruction'],
    num_rows: 26858
})

In [6]:
ds[0]

{'output': '以下是保持健康的三个提示：\n\n1. 保持身体活动。每天做适当的身体运动，如散步、跑步或游泳，能促进心血管健康，增强肌肉力量，并有助于减少体重。\n\n2. 均衡饮食。每天食用新鲜的蔬菜、水果、全谷物和脂肪含量低的蛋白质食物，避免高糖、高脂肪和加工食品，以保持健康的饮食习惯。\n\n3. 睡眠充足。睡眠对人体健康至关重要，成年人每天应保证 7-8 小时的睡眠。良好的睡眠有助于减轻压力，促进身体恢复，并提高注意力和记忆力。',
 'input': '',
 'instruction': '保持健康的三个提示。'}

In [7]:
tokenizer = AutoTokenizer.from_pretrained("Langboat/bloom-389m-zh")

In [17]:
def process_func(example):
    max_length = 256
    input_ids, attention_mask, labels = [] , [] , []
    instruction = tokenizer("\n".join(["Human: " + example['instruction'],example['input']]).strip() + "\n\nAssistant: ")
    response = tokenizer(example['output'] + tokenizer.eos_token)
    input_ids = instruction["input_ids"] + response["input_ids"]
    attention_mask = instruction["attention_mask"] + response["attention_mask"]
    labels = [-100] * len(instruction["input_ids"]) + response["input_ids"]
    if(len(input_ids) > max_length):
        input_ids = input_ids[:max_length]
        attention_mask = attention_mask[:max_length]
        labels = labels[:max_length]
    return {
        "input_ids":input_ids,
        "attention_mask":attention_mask,
        "labels":labels
    }

In [18]:
tokenized_ds = ds.map(process_func,remove_columns= ds.column_names)
tokenized_ds

Map:   0%|          | 0/26858 [00:00<?, ? examples/s]

Map: 100%|██████████| 26858/26858 [00:06<00:00, 3862.61 examples/s]


Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 26858
})

In [19]:
tokenizer.decode(tokenized_ds[0]["input_ids"])

'Human: 保持健康的三个提示。\n\nAssistant: 以下是保持健康的三个提示：\n\n1. 保持身体活动。每天做适当的身体运动，如散步、跑步或游泳，能促进心血管健康，增强肌肉力量，并有助于减少体重。\n\n2. 均衡饮食。每天食用新鲜的蔬菜、水果、全谷物和脂肪含量低的蛋白质食物，避免高糖、高脂肪和加工食品，以保持健康的饮食习惯。\n\n3. 睡眠充足。睡眠对人体健康至关重要，成年人每天应保证 7-8 小时的睡眠。良好的睡眠有助于减轻压力，促进身体恢复，并提高注意力和记忆力。</s>'

In [20]:
tokenized_ds[0]["labels"]

[-100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 4744,
 583,
 6583,
 24772,
 8995,
 13533,
 1022,
 189,
 189,
 20,
 17,
 210,
 6583,
 8416,
 3228,
 420,
 8634,
 1900,
 13648,
 8416,
 5625,
 355,
 1202,
 29011,
 553,
 30355,
 1298,
 15599,
 355,
 961,
 4872,
 34650,
 5980,
 355,
 10915,
 15342,
 7761,
 355,
 1403,
 11472,
 6189,
 20465,
 671,
 189,
 21,
 17,
 210,
 20122,
 13660,
 420,
 8634,
 13869,
 20189,
 373,
 17070,
 553,
 16382,
 553,
 1204,
 6165,
 1430,
 641,
 14562,
 16130,
 24251,
 15502,
 7984,
 355,
 7981,
 1220,
 6538,
 553,
 1220,
 14562,
 641,
 13545,
 10249,
 355,
 714,
 6583,
 24772,
 13660,
 11297,
 671,
 189,
 22,
 17,
 210,
 17672,
 16272,
 420,
 17672,
 1063,
 13966,
 5980,
 18688,
 355,
 30645,
 8634,
 1638,
 7900,
 954,
 3779,
 210,
 38858,
 17672,
 420,
 14054,
 17672,
 11472,
 15375,
 10891,
 355,
 4872,
 8416,
 7442,
 355,
 1403,
 5323,
 4001,
 16885,
 14721,
 1249,
 420,
 2]

In [22]:
tokenizer.decode(list(filter(lambda x : x != -100,tokenized_ds[0]["labels"])))

'以下是保持健康的三个提示：\n\n1. 保持身体活动。每天做适当的身体运动，如散步、跑步或游泳，能促进心血管健康，增强肌肉力量，并有助于减少体重。\n\n2. 均衡饮食。每天食用新鲜的蔬菜、水果、全谷物和脂肪含量低的蛋白质食物，避免高糖、高脂肪和加工食品，以保持健康的饮食习惯。\n\n3. 睡眠充足。睡眠对人体健康至关重要，成年人每天应保证 7-8 小时的睡眠。良好的睡眠有助于减轻压力，促进身体恢复，并提高注意力和记忆力。</s>'

## 模型创建

In [23]:
model = AutoModelForCausalLM.from_pretrained("Langboat/bloom-389m-zh")

In [25]:
args = TrainingArguments(
    output_dir= "./chatbox",
    per_device_train_batch_size= 4,
    gradient_accumulation_steps= 8,
    logging_steps= 8,
    num_train_epochs= 2
)

In [26]:
trainer = Trainer(args= args,model=model, train_dataset= tokenized_ds,data_collator= DataCollatorForSeq2Seq(tokenizer=tokenizer,padding=True))

In [30]:
trainer.train()

c:\Users\32721\anaconda3\envs\transformers\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
8,3.131500
16,3.151100
24,3.180700
32,3.122900
40,3.114800
48,3.030400
56,2.933100
64,2.940300
72,2.869500
80,2.870600


KeyboardInterrupt: 

## 推理

In [27]:
from transformers import pipeline

In [28]:
pipe = pipeline("text-generation",model=model,tokenizer=tokenizer)

Device set to use cpu


In [31]:
prompt = "Human: {}\n{}".format("考试有哪些技巧？","").strip() + "\n\nAssistant: "
pipe(prompt,max_length = 256)

[{'generated_text': 'Human: 考试有哪些技巧？\n\nAssistant: 考试技巧包括以下几点：\n\n1. 集中注意力：考试前应集中注意力，避免过多地思考或过多地集中注意力。\n\n2. 集中注意力：考试前应集中注意力，避免过多地思考或过多地集中注意力。\n\n3. 集中注意力：考试前应集中注意力，避免过多地思考或过多地集中注意力。\n\n4. 集中注意力：考试前应集中注意力，避免过多地思考或过多地集中注意力。\n\n5. 集中注意力：考试前应集中注意力，避免过多地思考或过多地集中注意力。\n\n6. 集中注意力：考试前应集中注意力，避免过多地思考或过多地集中注意力。\n\n7. 集中注意力：考试前应集中注意力，避免过多地思考或过多地集中注意力。\n\n8. 集中注意力：考试前应集中注意力，避免过多地思考或过多地集中注意力。\n\n9. 集中注意力：考试前应集中注意力，避免过多地思考或过多地集中注意力。\n\n10. 集中注意力：考试前应集中注意力，避免过多地思考或过多地集中注意力。'}]